In [140]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import root_mean_squared_error, r2_score
import joblib

In [145]:
df = pd.read_csv('../data/engineered_training_data.csv', parse_dates=['Time'])
df = df.set_index('Time')
df.head()

,GHI(W/m2),Windspeed(m/s),Solar Energy(MWh),Wind Energy(MWh),Total Energy(MWh),Battery Charge(MWh),Battery Discharge(MWh),Stored Energy(MWh),Energy To Grid(MWh),Hydrogen_yield(kg),...,Day,spline_hr_1,spline_hr_2,spline_hr_3,spline_hr_4,Windspeed_mean_3h,Windspeed_std_3h,GHI_mean_3h,Windspeed_lag_1hr,GHI_lag_1hr
Time,,,,,,,,,,,,,,,,,,,,,
2014-01-01 00:00:00,0.0,11.37,0.0,8.506240,8.506240,0.0,0.0,0.0,8.506240,159.891738,...,2,0.166667,0.666667,0.166667,0.000000,11.370000,0.000000,0.0,11.37,0.0
2014-01-01 01:00:00,0.0,11.22,0.0,8.174004,8.174004,0.0,0.0,0.0,8.174004,153.646687,...,2,0.093956,0.639051,0.266116,0.000877,11.295000,0.106066,0.0,11.37,0.0
2014-01-01 02:00:00,0.0,11.12,0.0,7.957390,7.957390,0.0,0.0,0.0,7.957390,149.574993,...,2,0.046232,0.566724,0.380031,0.007014,11.236667,0.125831,0.0,11.22,0.0
2014-01-01 03:00:00,0.0,10.67,0.0,7.029906,7.029906,0.0,0.0,0.0,7.029906,132.141091,...,2,0.018232,0.465467,0.492630,0.023671,11.003333,0.292973,0.0,11.12,0.0
2014-01-01 04:00:00,0.0,10.39,0.0,6.490870,6.490870,0.0,0.0,0.0,6.490870,122.008833,...,2,0.004699,0.351059,0.588135,0.056108,10.726667,0.368284,0.0,10.67,0.0


In [142]:
y_train = df['Hydrogen_yield(kg)']

# Using only the core weather features adding time and battery-relaated features will cause
# overfitting due to polynomial expansion which results in noise increasing the adjusted R2 score
# These 2 features give one of the highest Adjusted R2 scores further proving that keeping
# a baseline of these core features is the correct move
poly_features = ['Windspeed(m/s)' , 'GHI(W/m2)']
X_train_poly = df[poly_features]

print(f'No. of rows and columns in x: {X_train_poly.shape}')
print(f'No. of rows in y: {y_train.shape[0]}')

No. of rows and columns in x: (96432, 2)
No. of rows in y: 96432


In [143]:
# Wind Energy follows a cubic rule (windspeed ^ 3) which is why degree 3 works here the best
# Though a smooth polynomial curve does struggle when a cap of 9.5MW is kept prompting to try different models
poly = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly_trans = poly.fit_transform(X_train_poly)

poly_model = LinearRegression()
poly_model.fit(X_train_poly_trans, y_train)

y_train_pred = poly_model.predict(X_train_poly_trans)
rmse = root_mean_squared_error(y_train, y_train_pred)
r2 = r2_score(y_train, y_train_pred)

# Adjusted R2 score to observe the features which are noise
n = X_train_poly_trans.shape[0]
p = X_train_poly_trans.shape[1]
adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

print(f'Transformed Feature Count: {X_train_poly_trans.shape[1]}')
print(f'RMSE: {rmse:.3f} kg')
print(f'R2 Score: {r2:.4f}')
print(f'Adjusted R2 Score: {adjusted_r2:.4f}')

Transformed Feature Count: 9
RMSE: 18.524 kg
R2 Score: 0.8926
Adjusted R2 Score: 0.8926


In [144]:
# Exported both transformer and model to use in testing and deployment phase
joblib.dump(poly, '../models/poly_trans.joblib')
joblib.dump(poly_model, '../models/poly_reg_model.joblib')

['../models/poly_reg_model.joblib']